# Forward Modelling - NEMES 2026 workshop

Does forward modelling using the **Colin27 template**

1. Load one SNIRF file
2. Register optodes to Colin27 scalp (landmarks added from digpts.txt)
3. Compute forward model sensitivity matrix via nirfaster (i.e. channel space -> surface space matrix)

For details on each step, see Cedalion docs:
- [Tutorial 1 – Heads and Forward Models](https://doc.ibs.tu-berlin.de/cedalion/doc/dev/examples/tutorial/1_heads_and_fwm.html)


In [ ]:
import glob
import os

import cedalion
import cedalion.dataclasses
import cedalion.dot
import cedalion.io
import cedalion.io.snirf
import cedalion.nirs
import cedalion.nirs.cw
import cedalion.vis.anatomy
import cedalion.vis.blocks as vbx
from cedalion import units

import pyvista as pv
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr

np.set_printoptions(suppress=True)

# What counts as a short channel?
DIST_THRESHOLD = 1.5 * units.cm 

snirf_file = "../data/example.snirf"
fwm_dir = "../data/forward_model"

## Load example data

Load data and plot montage.

In [ ]:
rec   = cedalion.io.snirf.read_snirf(snirf_file)[0]

amp = rec.get_timeseries()
print("Amplitude shape:", dict(amp.sizes))
print("Wavelengths (nm):", amp.wavelength.values)

# Plot montage
cedalion.vis.anatomy.montage.plot_montage3D(rec['amp'], rec.geo3d)

## Get template head model

`get_standard_headmodel("colin27")` downloads the MNI152 Colin27 template with pre-segmented brain / scalp surfaces

fNIRS optodes (sources, detectors) and landmarks are registered to the scalp surface via rigid + elastic alignment (`align_and_relax_to_scalp`).

In [ ]:
head_ijk = cedalion.dot.get_standard_headmodel("colin27")

print("Scalp vertices:", head_ijk.scalp)
print("Brain vertices:", head_ijk.brain)
print("Landmarks:", head_ijk.landmarks.label.values)

In [ ]:
# Register optodes: snap to Colin27 scalp using the cranial landmarks
geo3d = rec.geo3d
geo3d_snapped_ijk, alignment_details = head_ijk.align_and_relax_to_scalp(
    geo3d, amp
)
print("Snapped geo3d dims:", dict(geo3d_snapped_ijk.sizes))

head_ras = head_ijk.apply_transform(head_ijk.t_ijk2ras)
display(head_ras)

# Let's plot the montage on the head
plt = pv.Plotter()
vbx.plot_surface(plt, head_ijk.scalp, color="#4fce64", opacity=.1)
vbx.plot_labeled_points(plt, geo3d_snapped_ijk)
vbx.plot_labeled_points(
    plt, head_ijk.landmarks.sel(label=["Nz", "Iz", "Cz", "LPA", "RPA"]), color="y"
)
plt.show()

## Compute forward model via photon transport simulation

The sensitivity matrix **Adot** (dims: `channel × vertex × wavelength`) is computed with the [micro (fast and furious) version of NIRFASTer](https://github.com/milabuob/nirfaster-uFF). This is the matrix that models sensitivity from each channel into cortical surface space.

In [ ]:
fluence_file     = f"{fwm_dir}/ParkMOVE_colin27_fluence.h5"
sensitivity_file = f"{fwm_dir}/ParkMOVE_colin27_Adot.h5"

if not os.path.exists(sensitivity_file):
    print("Computing forward model...")
    meas_list = rec._measurement_lists["amp"]
    fwm = cedalion.dot.ForwardModel(head_ijk, geo3d_snapped_ijk, meas_list)

    if not os.path.exists(fluence_file):
        fwm.compute_fluence_nirfaster(fluence_file)

    fwm.compute_sensitivity(fluence_file, sensitivity_file)
    print(f"Saved to {sensitivity_file}")
else:
    print(f"Loading cached sensitivity: {sensitivity_file}")

Adot = cedalion.io.load_Adot(sensitivity_file)
print("Adot dims:", dict(Adot.sizes))

## Visualize sensitivity matrix

The simulation also modelled the short channels so we identify those and only select the long channels.

In [ ]:
# Split long / short channels
amp_long, amp_short = cedalion.nirs.split_long_short_channels(
    amp, rec.geo3d, DIST_THRESHOLD
)
print(f"Long channels: {amp_long.sizes['channel']}")
print(f"Short channels: {amp_short.sizes['channel']}")

# Restrict Adot to long channels
shared_channels = np.intersect1d(Adot.channel.values, amp_long.channel.values)
Adot_long = Adot.sel(channel=shared_channels)

print(f"Long channels in Adot: {len(shared_channels)}")

In [ ]:
# Select only a subset of labeled points to plot
# Here we select sources and detectors via their labels that start with S or D
# I.e. skip landmarks and channel centers
geo3d_plot_ijk = geo3d_snapped_ijk.sel(
    label=geo3d_snapped_ijk.label.str.contains("S|D")
)

plotter = cedalion.vis.anatomy.sensitivity_matrix.Main(
    sensitivity=Adot_long,
    brain_surface=head_ijk.brain,
    head_surface=head_ijk.scalp,
    labeled_points= geo3d_plot_ijk,
)
plotter.plot(high_th=0, low_th=-2)
plotter.plt.show()